# P72 — IA neuro-simbólica: la tercera ola

## 1. Título y paper

**Paper:** *Neurosymbolic AI: The 3rd Wave*  
**Autoría:** Artur d'Avila Garcez, Luis C. Lamb  
**Año y venue:** 2020 · arXiv:2012.05876 · Artificial Intelligence Review (2023)  
**Nivel:** L5 · **Motor:** `neurosimbolico`  
**Ficha completa:** [`P72_neurosimbolico`](../../papers/foundational/P72_neurosimbolico/README.md)

**Hito:** Ordena la agenda de integrar aprendizaje y razonamiento en vez de elegir uno de los dos.

- [arXiv:2012.05876](https://arxiv.org/abs/2012.05876)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Las redes profundas aprenden de datos pero no razonan con reglas ni explican; los sistemas simbólicos razonan y explican pero no aprenden de datos ruidosos. Cada tradición tiene exactamente el punto ciego de la otra.
2. Ejecutar una implementación mínima de la propuesta: Una hoja de ruta para sistemas donde la percepción estima y los símbolos restringen, con requisitos explícitos: representación, aprendizaje, razonamiento y explicación en un mismo sistema.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P69
- P58
- Garcez, Broda y Gabbay (2002), sistemas neuro-simbólicos


## 4. Intuición

Una red estima y a veces se equivoca con confianza. Una regla no estima nada, pero sabe que en un salón no hay coches. Juntar las dos no es hacer una media: es dejar que la regla filtre el espacio de salidas de la red. Funciona muy bien cuando la regla es cierta, y destruye cuando no lo es.


## 5. Concepto mínimo

```text
percepción → distribución sobre etiquetas
restricción → elimina las etiquetas incompatibles con el contexto
decisión    → argmax sobre lo que queda

escena(salón) ∧ etiqueta(x, coche) → ⊥

No hay reentrenamiento: hay filtrado del espacio de salida.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('neurosimbolico', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos objetos del salón acierta la percepción sola?
2. ¿Y añadiendo la regla del contexto?
3. ¿Qué pasa si se aplica esa misma regla en un garaje?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('neurosimbolico', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('neurosimbolico', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

En el salón la percepción sola acierta **2 de 4** y con la regla, **4 de 4**: la restricción corrige dos objetos donde la red se equivocaba con confianza 0,55 y 0,48. En el garaje —donde sí hay coches— la misma regla es falsa y el resultado cae de **2 a 0 de 2**.


## 10. Comentario pedagógico

Esa asimetría es la tesis, y es lo que hace difícil el enfoque: el conocimiento simbólico aporta mucho cuando es correcto y cuesta todo cuando no lo es. De ahí que el requisito central no sea la integración técnica sino que las reglas estén **declaradas y sean auditables**, que es exactamente la ventaja que tenía [MYCIN](../../papers/foundational/P69_mycin/README.md) en 1975.


## 11. Error o anti-patrón deliberado

Anti-patrón: tratar el artículo como un método con resultados comparables.


In [ ]:
print('Garcez y Lamb publican un manifiesto y una hoja de ruta, no un sistema evaluado.')
print('No hay tabla de resultados que reproducir ni benchmark que superar.')
print('Se lee como agenda abierta, con fecha, no como estado del arte cerrado.')

## 12. Corrección

Lo que la miniatura sí demuestra:


In [ ]:
r = run_paper_lab('neurosimbolico', seed=7)['result']
print('salon  :', r['escena_salon']['aciertos'], '<- la regla vale')
print('garaje :', r['escena_garaje']['aciertos'], '<- la regla NO vale')
print('ganancia', r['ganancia_donde_la_regla_vale'], '| coste', r['coste_donde_no_vale'])

## 13. Desafío guiado

Localiza en la salida los dos objetos que la regla corrige y comprueba con qué confianza se equivocaba la percepción en cada uno.


In [ ]:
r = run_paper_lab('neurosimbolico', seed=3)['result']
show(r)

## 14. Desafío autónomo

Coge un clasificador tuyo, escribe dos restricciones de dominio que sepas ciertas y aplícalas sobre sus salidas. Mide la ganancia, y después busca deliberadamente un caso donde la restricción sea falsa y documenta el coste.


## 15. Evidencia de aprendizaje

Guarda la comparación entre las dos escenas y tu enunciado de por qué una restricción incorrecta no degrada sino que destruye.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P72_neurosimbolico/README.md) · evaluación formal: [`assessments/papers/P72_neurosimbolico.md`](../../assessments/papers/P72_neurosimbolico.md)


## 16. Cierre

Aquí se cierra la ruta simbólica. Lo que viene es la otra tradición: aprender la regla de los datos en vez de escribirla, que es lo que hace el machine learning clásico.


## 17. Conexión con el siguiente hito

- P28
- P29

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
